> 在线讲义：[第 23 章 · 架构演进与专用架构：每一代新增的东西，省掉了哪一笔](https://wikieden.github.io/ai-infra-from-constraints/lessons/23-architecture-evolution.html) · 本书《从约束推导 AI Infra》

# 第 23 章 · 架构演进与专用架构：每一代新增的东西，省掉了哪一笔

配套讲义：仓库里 `ai-infra/notes/lessons/23-architecture-evolution.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 V100 | $F/P$、$V/R$ | `v100_paths(m)` | 8.75 / 1.10 / 0.112 ms；2.14 / 0.268 / 37.3 μs |
| 第二步 MX | 位数 + scale | `weight_mib(bits)`、`read_us(mib, bw)` | 32 / 16.5 / 8.5 MiB；33.29 → 4.97 μs |
| 第三步 昇腾 | 字节 ÷ 时限 | `handoff_gbps(256, 1.024)` | 256 / 128 GB/s；2.79 μs |
| 第五步 Groq | 上取整 | `chips_needed(gib)` | 72 / 6 / 42 / 168 颗 |
| 第六步 ROM | $1 + W/(BK)$ | `rom_ratio(b)`、`lanes_needed(...)` | 13.5 / 1.8 倍；75,674 / 315,306 条 |

V100、MX 格式、产品线表、昇腾交接、Apple 统一内存、Groq SRAM、固定权重 ROM 的参数取自 AII 第 4 章 §4.6–§4.7。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
MiB = 2**20
GiB = 2**30

**函数 `v100_paths`**：V100 上 [m, d] × [d, d] 投影：(普通 FP32 μs, Tensor Core μs, FP16 读写 μs)。

In [ ]:
def v100_paths(m: int, d: int = 4096) -> tuple[float, float, float]:
    """V100 上 [m, d] × [d, d] 投影：(普通 FP32 μs, Tensor Core μs, FP16 读写 μs)。"""
    flops = 2 * m * d * d
    io = (m * d + d * d + m * d) * 2
    return flops / 15.7e12 * 1e6, flops / 125e12 * 1e6, io / 900e9 * 1e6

**函数 `weight_mib`**：d×d 权重的存储量（MiB）：BF16 无 scale，低精度每 group 个值共享 1 字节 scale。

In [ ]:
def weight_mib(bits: int, d: int = 4096, group: int = 32) -> float:
    """d×d 权重的存储量（MiB）：BF16 无 scale，低精度每 group 个值共享 1 字节 scale。"""
    n = d * d
    scale = 0 if bits == 16 else n / group
    return (n * bits / 8 + scale) / MiB

**函数 `read_us`**：小工具。

In [ ]:
def read_us(mib: float, bw: float) -> float:
    return mib * MiB / bw * 1e6

**函数 `handoff_gbps`**：在 budget_us 内传完 kib 所需带宽（GB/s）。

In [ ]:
def handoff_gbps(kib: float, budget_us: float) -> float:
    """在 budget_us 内传完 kib 所需带宽（GB/s）。"""
    return kib * 1024 / (budget_us * 1e-6) / 1e9

**函数 `chips_needed`**：小工具。

In [ ]:
def chips_needed(gib: float, sram_mib: float = 220) -> int:
    return math.ceil(gib * 1024 / sram_mib)

**函数 `rom_ratio`**：权重与 KV 共用 HBM 时的读取量 / 权重移到 ROM 后的 HBM 读取量 = 1 + W/(BK)。

In [ ]:
def rom_ratio(b: int, w: float = 15.1, k: float = 1.21) -> float:
    """权重与 KV 共用 HBM 时的读取量 / 权重移到 ROM 后的 HBM 读取量 = 1 + W/(BK)。"""
    return 1 + w / (b * k)

**函数 `lanes_needed`**：每条通道每周期一次 FMA（2 次运算），在 budget_s 内做完 ops 需要的通道数。

In [ ]:
def lanes_needed(ops: float, budget_s: float, util: float = 1.0, hz: float = 1e9) -> int:
    """每条通道每周期一次 FMA（2 次运算），在 budget_s 内做完 ops 需要的通道数。"""
    return math.ceil(ops / (2 * hz * budget_s * util))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：V100 上的 Q 投影")
for m in (4096, 1):
    a, b, io = v100_paths(m)
    print(f"  M = {m:4d}：普通 {a:,.2f} μs，Tensor Core {b:,.3f} μs，读写 {io:,.1f} μs")

In [ ]:
print("\n第二步：精度与带宽")
for bits, name in ((16, "BF16"), (8, "MXFP8"), (4, "MXFP4")):
    print(f"  {name}：{weight_mib(bits):.1f} MiB")
t4090, t5090, t5090_fp4 = read_us(32, 1008e9), read_us(32, 1792e9), read_us(8.5, 1792e9)
print(f"  4090 BF16 {t4090:.2f} μs → 5090 BF16 {t5090:.2f} μs → 5090 MXFP4 {t5090_fp4:.2f} μs，共 {t4090 / t5090_fp4:.2f} 倍")
print(f"  3090→4090 矩阵 {165.2 / 71.0:.2f} 倍、带宽 +{(1008 / 936 - 1) * 100:.1f}%；A100→H100 矩阵 {989.4 / 312:.2f} 倍、带宽 {3350 / 2039:.2f} 倍")
print(f"  64 MiB 过 NVLink：A100 {64 * MiB / 300e9 * 1e6:.0f} μs，H100 {64 * MiB / 450e9 * 1e6:.0f} μs")

In [ ]:
print("\n第三步：昇腾的两次改造")
unrolled = 3136 * 576 * 2 / MiB
print(f"  img2col 展开 {unrolled:.3f} MiB，HBM 写读 {2 * unrolled:.3f} MiB，{2 * unrolled * MiB / 1.2e12 * 1e6:.2f} μs")
print(f"  注意力块交接 256 KiB：94 GB/s 要 {256 * 1024 / 94e9 * 1e6:.2f} μs；1.024 μs 内完成要 {handoff_gbps(256, 1.024):.0f} GB/s；CV 直通 {handoff_gbps(128, 1.024):.0f} GB/s")

In [ ]:
print("\n第四步：Apple 统一内存")
for name, bw in (("M1/M2 Ultra", 800e9), ("M3 Ultra", 819e9), ("M4 Max", 546e9)):
    print(f"  Qwen3-8B 每步 16.345 GB：{name} {16.345e9 / bw * 1e3:.2f} ms")
print(f"  235B-A22B 每步 13.697 GB：M3 Ultra {13.697e9 / 819e9 * 1e3:.2f} ms；八请求 {8 / (24.737e9 / 819e9):.0f} / {8 / (75.967e9 / 819e9):.0f} token/s")

In [ ]:
print("\n第五步：Groq 每芯片 220 MiB SRAM")
print(f"  权重 16.4 GB → {chips_needed(16.4e9 / GiB)} 颗；KV 1 条 {chips_needed(1.125)}，8 条 {chips_needed(9)}，8 条 32K {chips_needed(36)}")

In [ ]:
print("\n第六步：固定权重放 ROM")
print(f"  HBM 读取缩小 B=1 {rom_ratio(1):.1f} 倍，B=16 {rom_ratio(16):.1f} 倍；交点 B = {15.1 / 1.21:.1f}，32K 时 {15.1 / (4 * 1.21):.2f}")
print(f"  100 μs/token 读 1.21 GB KV 要 {1.21e9 / 100e-6 / 1e12:.1f} TB/s")
ops = 15_134_641_792
print(f"  通道：100 μs {lanes_needed(ops, 100e-6):,}；40 μs、60% {lanes_needed(ops, 40e-6, 0.6):,}；256 条要 {ops / (2 * 256 * 1e9) * 1e3:.1f} ms")

## 3. 代码里才看得到的

1. V100 的 4,096 行读写是 $(2 \times 4096^2 + 4096^2) \times 2 = 96$ MiB，除以 900 GB/s 得 0.112 ms；单行时只剩权重 32 MiB 加两行输入输出。
2. 32K 时交点代码算出 3.12，原文写约 3.13：原文的 $K$ 没有取整到 1.21 GB。
3. Groq 的 72 颗按 16.4 GB 权重（$10^9$ 制）除以 220 MiB（$2^{20}$ 制）得 71.1，再上取整。

## 4. 自己改一改

- 把 `v100_paths` 改成 H100（BF16 989.4 TFLOP/s、3.35 TB/s），求计算和读写相等的行数。
- 给 `rom_ratio` 加一个上下文参数，画出 8K / 32K / 128K 三条曲线。
- 用 `chips_needed` 算 Cerebras WSE-3（44 GB）能放几条 8K 请求的 KV。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 灶快只帮大锅菜，一份菜还得跑仓库；
> 小字号省纸，三十二字配个放大镜；
> 仓库大能多放，出货口宽才出得快；
> 机床旁货架不能互借，作业本越写越厚；
> 书印在墙上，人一多书包照样重。